In [14]:
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [2]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset



In [3]:
class CityDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        """
        Args:
            csv_file (str): CSV dosyasının yolu (ör. 'train.csv').
            img_dir (str): Görsellerin bulunduğu klasörün yolu (ör. 'train/').
            transform (callable, optional): Görsellere uygulanacak dönüşümler.
        """
        self.annotations = pd.read_csv(csv_file)  # CSV'den bilgileri yükle
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        # Toplam veri sayısını döndür
        return len(self.annotations)

    def __getitem__(self, idx):
        # Belirtilen indeksteki veriyi döndür
        img_path = os.path.join(self.img_dir, self.annotations.iloc[idx, 0])  # Görselin dosya yolu
        image = Image.open(img_path).convert("RGB")  # Görseli yükle ve RGB formatına çevir
        label = self.annotations.iloc[idx, 1]  # Şehrin adı (ör. Istanbul, Ankara, Izmir)
        
        # Şehir isimlerini sınıf etiketlerine dönüştür (ör. Istanbul -> 0, Ankara -> 1, Izmir -> 2)
        label_mapping = {"Istanbul": 0, "Ankara": 1, "Izmir": 2}
        label = label_mapping[label]

        if self.transform:
            image = self.transform(image)  # Görsele dönüşüm uygula

        return image, label


In [4]:
from torchvision import transforms
from torch.utils.data import DataLoader

# Dönüşümleri tanımlayın
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Görselleri 224x224 boyutuna yeniden boyutlandır
    transforms.ToTensor(),  # Görselleri Tensor formatına dönüştür
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalizasyon
])

# Dataset ve DataLoader
train_dataset = CityDataset(csv_file='train_data.csv', img_dir='train\\train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


In [8]:
for images, labels in train_loader:
    print(images.shape)  # Görsellerin boyutunu kontrol edin (ör. [32, 3, 224, 224])
    print(labels)  # Etiketlerin doğru olup olmadığını kontrol edin
    break


torch.Size([32, 3, 224, 224])
tensor([2, 0, 2, 1, 0, 1, 1, 2, 0, 0, 2, 2, 0, 1, 2, 1, 0, 2, 1, 0, 2, 1, 2, 0,
        0, 2, 2, 2, 0, 0, 2, 2])


In [10]:
from sklearn.model_selection import train_test_split

# CSV'yi bölme
data = pd.read_csv('train_data.csv')
train_data, val_data = train_test_split(data, test_size=0.2, stratify=data['city'])

# Bölünmüş CSV'leri kaydetme (isteğe bağlı)
train_data.to_csv('train_split.csv', index=False)
val_data.to_csv('val_split.csv', index=False)

# Dataset oluşturma
train_dataset = CityDataset(csv_file='train_split.csv', img_dir='train\\train', transform=transform)
val_dataset = CityDataset(csv_file='val_split.csv', img_dir='train\\train', transform=transform)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [12]:
num_epochs = 10  # Modeli 10 epoch boyunca eğiteceğiz

In [14]:
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Doğrulama
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"Epoch {epoch+1}/{num_epochs}, F1 Score: {f1:.4f}")
